# Biểu đồ đánh giá DRL (PPO/SAC) cho báo cáo đồ án

Phiên bản notebook của `plot_metrics.py` — dùng khi muốn xem trước từng biểu đồ ngay trong Jupyter
trước khi nhúng vào báo cáo, thay vì chạy CLI rồi mở file ảnh riêng. Hai bản **dùng chung toàn bộ logic**
(hàm vẽ, bảng màu, style) từ `plot_metrics.py` — notebook này chỉ gọi lại các hàm đó với `show=True`
để hiển thị inline, không viết lại bất kỳ phép vẽ nào — sửa logic vẫn chỉ sửa ở một nơi.

**Không cần CARLA/torch** — chỉ `numpy` + `matplotlib`, chạy được trên máy viết báo cáo (khác máy train).

**Cách dùng**: mở notebook này từ thư mục `drl_training/` (để import `plot_metrics.py` cạnh nó), sửa
đường dẫn ở cell **Cấu hình** bên dưới cho khớp thư mục `runs/...` thật của bạn, rồi **Run All**. Mỗi
biểu đồ vừa hiện inline vừa được lưu ra `--output` (`.png` cho Word + `.pdf` vector cho LaTeX/Overleaf).

Chỉ có 1 thuật toán (ví dụ mới train PPO)? Để trống `SAC_DIR`/`EVAL_SAC_CSV` = `None` — các cell
liên quan SAC/so sánh 2 thuật toán sẽ tự in cảnh báo và bỏ qua, không lỗi.

In [ ]:
%matplotlib inline

import sys
from pathlib import Path

# Gia dinh notebook duoc mo/chay tu thu muc drl_training/ (canh plot_metrics.py). Neu
# khong, sua NOTEBOOK_DIR thanh duong dan tuyet doi toi drl_training/.
NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from plot_metrics import (  # noqa: E402 - tai su dung toan bo logic ve tu plot_metrics.py
    COLOR_PPO, COLOR_SAC, Run, _apply_style,
    plot_learning_curve, plot_update_diagnostic, plot_terminate_reason,
    plot_eval_comparison, plot_throughput,
)

_apply_style()
print("Da nap plot_metrics.py tu:", NOTEBOOK_DIR / "plot_metrics.py")

## Cấu hình — sửa các đường dẫn này cho khớp lần train thật của bạn

`PPO_DIR`/`SAC_DIR` = thư mục `output` của `train_ppo.py`/`train_sac.py` (chứa `episode_log.csv` +
`update_log.csv`). `EVAL_*_CSV` = file từ `evaluate.py --eval-csv-out ...` (tùy chọn, chỉ cần cho
biểu đồ so sánh cuối). `IL_MAE`/`IL_COLLISION_RATE` = số liệu riêng của bước IL (mục 11 notebook IL),
dùng làm đường baseline tham chiếu — để `None` nếu chưa có.

In [ ]:
PPO_DIR = "runs/ppo_lane_keep"          # None neu chua train PPO
SAC_DIR = "runs/sac_lane_keep"          # None neu chua train SAC

EVAL_PPO_CSV = "runs/ppo_lane_keep/eval_results.csv"   # None neu chua chay evaluate.py --eval-csv-out
EVAL_SAC_CSV = "runs/sac_lane_keep/eval_results.csv"   # None neu chua chay evaluate.py --eval-csv-out

IL_MAE = None              # vd 0.25 - lech lan trung binh |m| cua rieng buoc IL, neu co
IL_COLLISION_RATE = None   # vd 22.0 - ti le va cham (%) cua rieng buoc IL, neu co

REWARD_WINDOW = 20   # do rong cua so trung binh truot cho duong hoc (khop 'recent_episode_rewards' trong train_*.py)
OUTPUT_DIR = Path("./report_figures").expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
runs = [
    Run("PPO", COLOR_PPO, run_dir=PPO_DIR, eval_csv=EVAL_PPO_CSV),
    Run("SAC", COLOR_SAC, run_dir=SAC_DIR, eval_csv=EVAL_SAC_CSV),
]
runs = [r for r in runs if r.episode_rows or r.update_rows or r.eval_rows]
ppo_run = next((r for r in runs if r.name == "PPO" and r.update_rows), None)
sac_run = next((r for r in runs if r.name == "SAC" and r.update_rows), None)

if not runs:
    print("[!] Khong doc duoc du lieu nao tu PPO_DIR/SAC_DIR o cell Cau hinh — kiem tra lai duong dan.")
else:
    print("Cac lan chay doc duoc:", ", ".join(r.name for r in runs))
print("Bieu do se duoc luu vao:", OUTPUT_DIR)

## 1. Đường học — Reward theo episode

Nét nhạt = reward thô từng episode, nét đậm = trung bình trượt `REWARD_WINDOW` episode gần nhất.

In [ ]:
plot_learning_curve(runs, "episode_reward", "Reward / episode", "Duong hoc — Reward theo episode",
                     "01_reward_curve", OUTPUT_DIR, REWARD_WINDOW, show=True)

## 2. Đường học — Độ dài episode

Episode dài dần theo thời gian train thường là dấu hiệu policy ít va chạm/lệch làn sớm hơn.

In [ ]:
plot_learning_curve(runs, "episode_len", "So buoc / episode", "Duong hoc — Do dai episode",
                     "02_episode_length", OUTPUT_DIR, REWARD_WINDOW, show=True)

## 3. Lý do kết thúc episode theo tiến trình train

Cột xếp chồng 100% theo từng “cửa sổ” episode liên tiếp — muốn thấy phần **đỏ (va chạm)** thu hẹp và
phần **xanh (hết giờ, an toàn)** mở rộng dần về cuối quá trình train.

In [ ]:
plot_terminate_reason(runs, "03_terminate_reason", OUTPUT_DIR, show=True)

## 4–5. Chẩn đoán PPO (clipped surrogate)

`approx_kl` nên dời quanh `target_kl=0.02` (đường gạch ngang) — liên tục vượt xa nguỡng này là dấu
hiệu learning rate quá cao. Bỏ qua nếu chưa train PPO.

In [ ]:
if ppo_run:
    plot_update_diagnostic([ppo_run],
        [("policy_loss", "Policy loss", None), ("value_loss", "Value loss", None)],
        "PPO — Policy loss & Value loss", "04_ppo_losses", OUTPUT_DIR, REWARD_WINDOW, show=True)

    TARGET_KL = 0.02
    plot_update_diagnostic([ppo_run],
        [("approx_kl", "Approx. KL", (TARGET_KL, "target_kl=%.2f" % TARGET_KL)),
         ("clip_fraction", "Ti le bi clip", None),
         ("entropy", "Entropy chinh sach", None)],
        "PPO — Chan doan huan luyen", "05_ppo_diagnostics", OUTPUT_DIR, REWARD_WINDOW, show=True)
else:
    print("[!] Chua co du lieu PPO (PPO_DIR trong hoac thieu update_log.csv) — bo qua.")

## 6–7. Chẩn đoán SAC (twin-Q + auto temperature)

`alpha` (temperature) thường giảm dần khi chính sách ổn định; `mean_q` tăng bất thường (rất lớn hoặc
âm sâu) là dấu hiệu overestimation. Bỏ qua nếu chưa train SAC.

In [ ]:
if sac_run:
    plot_update_diagnostic([sac_run],
        [("critic_loss", "Critic loss", None), ("actor_loss", "Actor loss", None)],
        "SAC — Critic loss & Actor loss", "06_sac_losses", OUTPUT_DIR, REWARD_WINDOW, show=True)

    plot_update_diagnostic([sac_run],
        [("alpha", "Temperature (alpha)", None), ("mean_q", "Mean Q", None),
         ("entropy", "Entropy chinh sach", None)],
        "SAC — Chan doan huan luyen", "07_sac_diagnostics", OUTPUT_DIR, REWARD_WINDOW, show=True)
else:
    print("[!] Chua co du lieu SAC (SAC_DIR trong hoac thieu update_log.csv) — bo qua.")

## 8. Thông lượng huấn luyện

`steps_per_sec` — biểu đồ phụ trợ đánh giá hiệu năng hệ thống (không phải chất lượng policy).

In [ ]:
plot_throughput(runs, "08_throughput", OUTPUT_DIR, REWARD_WINDOW, show=True)

## 9. So sánh đánh giá cuối (PPO vs SAC vs IL)

Cần `EVAL_PPO_CSV`/`EVAL_SAC_CSV` (từ `evaluate.py --eval-csv-out ... --deterministic`). `IL_MAE`/
`IL_COLLISION_RATE` chỉ xuất hiện nếu được điền ở cell Cấu hình.

In [ ]:
plot_eval_comparison(runs, IL_MAE, IL_COLLISION_RATE, "09_eval_comparison", OUTPUT_DIR, show=True)

## Tổng kết

Mỗi biểu đồ ở trên đã được lưu vào `OUTPUT_DIR` ở cả `.png` (nhúng vào Word) và `.pdf` (vector, nhúng vào
LaTeX/Overleaf). Danh sách file:

In [ ]:
for path in sorted(OUTPUT_DIR.glob("*.png")):
    print(path.name)